In [1]:
# ================================================================
# CELL 1 — Imports & Configuration
# ================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, roc_curve, confusion_matrix
)

# ── Hyperparameters ──────────────────────────────────────────────
N_SPLITS   = 3          # number of k-fold splits
EPOCHS     = 5         # max epochs per fold
PATIENCE   = 5          # early-stopping patience
BATCH_SIZE = 32
LR         = 5e-4
RANDOM_STATE = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"K-Fold splits: {N_SPLITS}  |  Max epochs/fold: {EPOCHS}  |  Patience: {PATIENCE}")

Device: cpu
K-Fold splits: 3  |  Max epochs/fold: 5  |  Patience: 5


In [2]:
# ================================================================
# CELL 2 — Load Data
# ================================================================
X = np.load("X_features.npy")   # (N, 128, 94, 1)
y = np.load("y_labels.npy")     # (N,)

print(f"Dataset loaded  →  X: {X.shape}  |  y: {y.shape}")
print(f"Class distribution  →  0: {(y==0).sum()}  |  1: {(y==1).sum()}")

Dataset loaded  →  X: (18105, 128, 94, 1)  |  y: (18105,)
Class distribution  →  0: 7172  |  1: 10933


In [3]:
# ================================================================
# CELL 3 — Dataset Class & Model Architecture
# ================================================================

class AudioDataset(Dataset):
    """Wraps numpy arrays; permutes (N,H,W,C) → (N,C,H,W) for PyTorch."""
    def __init__(self, features, labels):
        self.X = torch.from_numpy(features).permute(0, 3, 1, 2).float()
        self.y = torch.from_numpy(labels).float().unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class DeepfakeDetector(nn.Module):
    """CNN deepfake detector on Log-Mel spectrograms."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((8, 8))
        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.adaptive_pool(x)
        return self.fc(torch.flatten(x, 1))


print("AudioDataset and DeepfakeDetector defined.")

AudioDataset and DeepfakeDetector defined.


In [4]:
# ================================================================
# CELL 4 — Helper Functions
# ================================================================

def get_probs(model, loader):
    """Run model in eval mode; return raw probabilities and true labels."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            p = torch.sigmoid(model(bx)).cpu().numpy().ravel()
            probs.extend(p)
            labels.extend(by.numpy().ravel())
    return np.array(probs), np.array(labels)


def youden_threshold(probs, labels):
    """Find the threshold that maximises Youden's J statistic (TPR - FPR)."""
    fpr, tpr, thresholds = roc_curve(labels, probs)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    return float(thresholds[best_idx]), float(j_scores[best_idx])


def evaluate_with_threshold(probs, labels, threshold):
    """Return a dict of metrics given a fixed decision threshold."""
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    return {
        "threshold" : threshold,
        "accuracy"  : accuracy_score(labels, preds),
        "f1"        : f1_score(labels, preds, zero_division=0),
        "auc"       : roc_auc_score(labels, probs),
        "sensitivity": tp / (tp + fn + 1e-8),   # recall / TPR
        "specificity": tn / (tn + fp + 1e-8),   # TNR
        "precision" : tp / (tp + fp + 1e-8),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "report"    : classification_report(labels, preds, digits=4),
    }


print("Helper functions defined.")

Helper functions defined.


In [5]:
# ================================================================
# CELL 5 — Stratified K-Fold Training Loop
# ================================================================

full_dataset = AudioDataset(X, y)
skf          = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_results = []          # stores per-fold metric dicts
fold_thresholds = []       # Youden thresholds per fold
best_global_auc = -1
best_model_state = None

print("="*65)
print(f"  Starting {N_SPLITS}-Fold Cross-Validation")
print("="*65)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):

    print(f"\n{'─'*65}")
    print(f"  FOLD {fold_idx} / {N_SPLITS}   "
          f"  train={len(train_idx)}  val={len(val_idx)}")
    print(f"{'─'*65}")

    # ── DataLoaders ──────────────────────────────────────────────
    train_loader = DataLoader(
        full_dataset,
        batch_size=BATCH_SIZE,
        sampler=SubsetRandomSampler(train_idx)
    )
    val_loader = DataLoader(
        full_dataset,
        batch_size=BATCH_SIZE,
        sampler=SubsetRandomSampler(val_idx)
    )

    # ── Fresh model per fold ──────────────────────────────────────
    model     = DeepfakeDetector().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    best_val_auc = -1
    best_fold_state = None
    patience_counter = 0
    epochs_run = 0

    # ── Training ─────────────────────────────────────────────────
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        # ── Validate ─────────────────────────────────────────────
        val_probs, val_labels = get_probs(model, val_loader)
        thresh, _  = youden_threshold(val_probs, val_labels)
        val_preds  = (val_probs >= thresh).astype(int)
        val_acc    = accuracy_score(val_labels, val_preds)
        val_auc    = roc_auc_score(val_labels, val_probs)

        print(f"  Epoch {epoch:02d}/{EPOCHS}  "
              f"| Loss: {train_loss:.4f}  "
              f"| Val Acc: {val_acc:.4f}  "
              f"| Val AUC: {val_auc:.4f}  "
              f"| Youden Thresh: {thresh:.4f}")

        epochs_run += 1

        # ── Early stopping on AUC ────────────────────────────────
        if val_auc > best_val_auc:
            best_val_auc   = val_auc
            best_fold_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  ⚠  Early stopping at epoch {epoch} "
                      f"(no AUC improvement for {PATIENCE} epochs)")
                break

    # ── Load best checkpoint for this fold ───────────────────────
    model.load_state_dict(best_fold_state)

    # ── Final fold evaluation with Youden threshold ───────────────
    val_probs, val_labels = get_probs(model, val_loader)
    youden_thresh, j_stat = youden_threshold(val_probs, val_labels)
    metrics = evaluate_with_threshold(val_probs, val_labels, youden_thresh)
    metrics["fold"]       = fold_idx
    metrics["epochs_run"] = epochs_run
    metrics["j_stat"]     = j_stat

    fold_results.append(metrics)
    fold_thresholds.append(youden_thresh)

    print(f"\n  ✦ Fold {fold_idx} Summary (Youden threshold = {youden_thresh:.4f})")
    print(f"    Epochs run : {epochs_run}")
    print(f"    Accuracy   : {metrics['accuracy']:.4f}")
    print(f"    F1 Score   : {metrics['f1']:.4f}")
    print(f"    AUC-ROC    : {metrics['auc']:.4f}")
    print(f"    Sensitivity: {metrics['sensitivity']:.4f}")
    print(f"    Specificity: {metrics['specificity']:.4f}")
    print(f"    Youden J   : {j_stat:.4f}")

    # ── Track globally best model ─────────────────────────────────
    if metrics["auc"] > best_global_auc:
        best_global_auc  = metrics["auc"]
        best_model_state = best_fold_state
        best_fold_idx    = fold_idx
        best_youden_thresh = youden_thresh

print("\n" + "="*65)
print(f"  All {N_SPLITS} folds complete.")
print("="*65)

  Starting 3-Fold Cross-Validation

─────────────────────────────────────────────────────────────────
  FOLD 1 / 3     train=12070  val=6035
─────────────────────────────────────────────────────────────────
  Epoch 01/5  | Loss: 0.4378  | Val Acc: 0.8464  | Val AUC: 0.9306  | Youden Thresh: 0.3734
  Epoch 02/5  | Loss: 0.3291  | Val Acc: 0.8510  | Val AUC: 0.9351  | Youden Thresh: 0.9240
  Epoch 03/5  | Loss: 0.2680  | Val Acc: 0.9185  | Val AUC: 0.9741  | Youden Thresh: 0.2933
  Epoch 04/5  | Loss: 0.2269  | Val Acc: 0.9385  | Val AUC: 0.9851  | Youden Thresh: 0.2863
  Epoch 05/5  | Loss: 0.1982  | Val Acc: 0.9064  | Val AUC: 0.9618  | Youden Thresh: 0.0501

  ✦ Fold 1 Summary (Youden threshold = 0.2863)
    Epochs run : 5
    Accuracy   : 0.9385
    F1 Score   : 0.9491
    AUC-ROC    : 0.9851
    Sensitivity: 0.9495
    Specificity: 0.9218
    Youden J   : 0.8713

─────────────────────────────────────────────────────────────────
  FOLD 2 / 3     train=12070  val=6035
────────────────

In [6]:
# ================================================================
# CELL 6 — Cross-Validation Summary Metrics
# ================================================================

metrics_keys = ["accuracy", "f1", "auc", "sensitivity", "specificity", "threshold", "j_stat"]

print("\n" + "="*65)
print("  K-FOLD CROSS-VALIDATION RESULTS")
print("="*65)
print(f"  Total iterations (folds): {N_SPLITS}")
print(f"  Threshold method        : Youden's J statistic (TPR − FPR)")
print()

# Per-fold table
header = f"{'Fold':>5} {'Epochs':>7} {'Thresh':>8} {'Acc':>8} {'F1':>8} {'AUC':>8} {'Sens':>8} {'Spec':>8} {'J-stat':>8}"
print(header)
print("-" * len(header))

for r in fold_results:
    print(f"{r['fold']:>5} "
          f"{r['epochs_run']:>7} "
          f"{r['threshold']:>8.4f} "
          f"{r['accuracy']:>8.4f} "
          f"{r['f1']:>8.4f} "
          f"{r['auc']:>8.4f} "
          f"{r['sensitivity']:>8.4f} "
          f"{r['specificity']:>8.4f} "
          f"{r['j_stat']:>8.4f}")

print("-" * len(header))

# Aggregate statistics
for k in metrics_keys:
    vals = [r[k] for r in fold_results]
    print(f"  {k:<14}  mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  "
          f"min={np.min(vals):.4f}  max={np.max(vals):.4f}")

print()
total_epochs = sum(r['epochs_run'] for r in fold_results)
print(f"  Total epochs trained across all folds : {total_epochs}")
print(f"  Best fold (highest AUC)               : Fold {best_fold_idx}  (AUC={best_global_auc:.4f})")
print(f"  Youden threshold for best fold        : {best_youden_thresh:.4f}")


  K-FOLD CROSS-VALIDATION RESULTS
  Total iterations (folds): 3
  Threshold method        : Youden's J statistic (TPR − FPR)

 Fold  Epochs   Thresh      Acc       F1      AUC     Sens     Spec   J-stat
----------------------------------------------------------------------------
    1       5   0.2863   0.9385   0.9491   0.9851   0.9495   0.9218   0.8713
    2       5   0.0357   0.9152   0.9300   0.9691   0.9333   0.8875   0.8208
    3       5   0.7596   0.9021   0.9174   0.9716   0.9009   0.9038   0.8047
----------------------------------------------------------------------------
  accuracy        mean=0.9186  std=0.0151  min=0.9021  max=0.9385
  f1              mean=0.9322  std=0.0130  min=0.9174  max=0.9491
  auc             mean=0.9753  std=0.0070  min=0.9691  max=0.9851
  sensitivity     mean=0.9279  std=0.0202  min=0.9009  max=0.9495
  specificity     mean=0.9044  std=0.0140  min=0.8875  max=0.9218
  threshold       mean=0.3605  std=0.3001  min=0.0357  max=0.7596
  j_stat       

In [7]:
# ================================================================
# CELL 7 — Full Classification Report for Best Fold
# ================================================================

best_fold_metrics = next(r for r in fold_results if r["fold"] == best_fold_idx)

print(f"\nDetailed Classification Report — Best Fold (Fold {best_fold_idx})")
print(f"Youden's Optimal Threshold = {best_fold_metrics['threshold']:.4f}")
print(f"Youden's J Statistic       = {best_fold_metrics['j_stat']:.4f}\n")
print(best_fold_metrics["report"])

print("Confusion Matrix (TP / FP / FN / TN):")
print(f"  TP={best_fold_metrics['tp']}  FP={best_fold_metrics['fp']}  "
      f"FN={best_fold_metrics['fn']}  TN={best_fold_metrics['tn']}")


Detailed Classification Report — Best Fold (Fold 1)
Youden's Optimal Threshold = 0.2863
Youden's J Statistic       = 0.8713

              precision    recall  f1-score   support

         0.0     0.9229    0.9218    0.9223      2390
         1.0     0.9487    0.9495    0.9491      3645

    accuracy                         0.9385      6035
   macro avg     0.9358    0.9356    0.9357      6035
weighted avg     0.9385    0.9385    0.9385      6035

Confusion Matrix (TP / FP / FN / TN):
  TP=3461  FP=187  FN=184  TN=2203


In [ ]:
# ================================================================
# CELL 8 — Save Best Model
# ================================================================

SAVE_PATH = "deepfake_logmel_model_kfold.pth"

torch.save(
    {
        "model_state_dict"   : best_model_state,
        "best_fold"          : best_fold_idx,
        "best_auc"           : best_global_auc,
        "youden_threshold"   : best_youden_thresh,
        "n_splits"           : N_SPLITS,
        "fold_results"       : [
            {k: v for k, v in r.items() if k != "report"}
            for r in fold_results
        ],
    },
    SAVE_PATH,
)

print(f"✅ Model checkpoint saved → {SAVE_PATH}")
print(f"   Best fold      : Fold {best_fold_idx}")
print(f"   Best AUC       : {best_global_auc:.4f}")
print(f"   Youden thresh  : {best_youden_thresh:.4f}")
print()
print("To reload:")
print("  ckpt = torch.load('deepfake_logmel_model_kfold.pth')")
print("  model.load_state_dict(ckpt['model_state_dict'])")
print("  threshold = ckpt['youden_threshold']")